# Retail Sales Analysis
Beginner-friendly data analysis using Pandas and Matplotlib. Dataset is **synthetic**.


## 1. Load the data

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
df = pd.read_csv(project_root / "data" / "sales_sample.csv")
df.head()

## 2. Explore and clean

In [ ]:
print(df.shape)
print(df.isnull().sum())
print(df.describe())
df = df.drop_duplicates(subset="order_id").copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
for col in ["quantity", "unit_price", "unit_cost", "discount"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["date", "product", "category", "city", "quantity", "unit_price", "unit_cost", "discount"])
df = df[(df["quantity"] > 0) & (df["unit_price"] >= 0) & (df["unit_cost"] >= 0) & df["discount"].between(0,1)].copy()

## 3. Calculate revenue and profit

In [ ]:
df["revenue"] = df["quantity"] * df["unit_price"] * (1 - df["discount"])
df["profit"] = df["revenue"] - df["quantity"] * df["unit_cost"]
df["month"] = df["date"].dt.to_period("M").astype(str)
print("Revenue:", round(df["revenue"].sum(), 2))
print("Profit:", round(df["profit"].sum(), 2))
print("Orders:", df["order_id"].nunique())

## 4. Analyze monthly revenue

In [ ]:
monthly = df.groupby("month")["revenue"].sum().sort_index()
print(monthly)
monthly.plot(kind="line", marker="o", figsize=(10,4), title="Monthly Revenue")
plt.ylabel("Revenue")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 5. Compare categories and products

In [ ]:
by_category = df.groupby("category")["revenue"].sum().sort_values(ascending=False)
print(by_category)
by_category.plot(kind="bar", figsize=(8,4), title="Revenue by Category")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

print(df.groupby("product")["quantity"].sum().sort_values(ascending=False).head(5))

## 6. Compare cities

In [ ]:
by_city = df.groupby("city")["revenue"].sum().sort_values(ascending=False)
print(by_city)
by_city.plot(kind="bar", title="Revenue by City")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

## 7. Write conclusions
Explain the highest-revenue month, best-selling product, leading category and city. Remember: this is synthetic data; insights are demonstrations only.